# 01 — Coleta e integração
Fonte: portal oficial de [Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf), arquivos **agrupados por ocorrência**. Os links são obtidos da página oficial para evitar URLs desatualizadas.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "outputs"
for p in [DATA/"raw", DATA/"processed", OUT/"auditoria", OUT/"eda", OUT/"estatistica", OUT/"modelos", OUT/"interpretabilidade", OUT/"modelo_final"]:
    p.mkdir(parents=True,exist_ok=True)
SILVER = DATA/"processed"/"prf_acidentes_silver_2020_2025.parquet"
import requests,zipfile,io,re
from bs4 import BeautifulSoup
URL="https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf"
resp=requests.get(URL,timeout=60)
resp.raise_for_status()
soup=BeautifulSoup(resp.text,"html.parser")
links={}
for tr in soup.select("tr"):
    txt=tr.get_text(" ",strip=True)
    for ano in range(2020,2026):
        if ("Acidentes" in txt and str(ano) in txt and "Agrupados por ocorrência" in txt):
            a=tr.find("a",href=True)
            if a: links[ano]=requests.compat.urljoin(URL,a["href"])
print(links)
assert len(links)==6, "Links não encontrados: confira a página da PRF ou preencha manualmente os 6 links oficiais antes de continuar."

In [ ]:
bases=[]
for ano,url in sorted(links.items()):
    pasta=DATA/"raw"/str(ano); pasta.mkdir(parents=True,exist_ok=True)
    r=requests.get(url,timeout=120);r.raise_for_status()
    content=r.content
    if zipfile.is_zipfile(io.BytesIO(content)):
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            csvs=[n for n in z.namelist() if n.lower().endswith(".csv")]
            assert csvs,f"Sem CSV no ZIP {ano}"
            raw=z.read(csvs[0])
    else: raw=content
    df_ano=None
    for enc in ("utf-8-sig","latin-1"):
        try: df_ano=pd.read_csv(io.BytesIO(raw),sep=";",encoding=enc,low_memory=False);break
        except UnicodeDecodeError: pass
    assert df_ano is not None,f"Falha na leitura {ano}"
    df_ano.columns=df_ano.columns.str.strip().str.lower()
    df_ano["ano_fonte"]=ano
    print(ano,len(df_ano))
    bases.append(df_ano)
df=pd.concat(bases,ignore_index=True)
for c in ("km","latitude","longitude"):
    df[c]=pd.to_numeric(df[c].astype("string").str.replace(",",".",regex=False),errors="coerce")
print("Registros:",len(df),"Colunas:",len(df.columns),"IDs repetidos:",df.id.duplicated().sum())
df.to_parquet(DATA/"processed"/"prf_acidentes_2020_2025.parquet",index=False)
